# Reproducing Mannion (2015) from CRCNS vim-3

**Paper.** Mannion DJ (2015). Sensitivity to the visual field origin of natural image patches in human
low-level visual cortex. *PeerJ* 3:e1038. [doi:10.7717/peerj.1038](https://doi.org/10.7717/peerj.1038).
The claims file (`claims/crcns-vim-3/10.7717-peerj.1038.yaml`) was written from the version of record
(PMC4485252); the supplementary files themselves were not available, only their legends.

**Data set.** [CRCNS vim-3](https://doi.org/10.6080/K0JS9NC2), doi:10.6080/K0JS9NC2, read from the public
bucket at `https://crcnsarchive.s3-us-west-2.amazonaws.com/vim-3/`. CRCNS data sets are not versioned
(docs/archives.md), so the version is recorded as `unversioned`, and the paper cites the data set by its
DOI only. The data set has one data file, `crcns_vim-3.tar.gz` (2,349,486,063 bytes), with md5
`005760a76158d4f16f54b41ee24cd1f2` in the data set's `checksums.md5`; the notebook streams the whole file
once, computes its md5 as it reads and compares it with that list below, so a later change of the file
is detected. The data descriptor is version 0.5 of 12 June 2015. There is no other version to choose.
A tar.gz has no index, so it cannot be opened in place as a zip can; it is streamed member by member and
nothing is written to disk.

**What the data set holds.** For each of the seven participants, the ten functional runs (NIfTI, not
preprocessed), the event sequence of each run, the digit stream of the central task and the button
presses; and screenshots of the 120 stimuli. It does not hold the anatomical images, the cortical
surfaces, the retinotopic mapping or the V1, V2 and V3 definitions; the descriptor says so ("You may
need additional data (anatomical images, visual area definitions, etc.) that are not part of the
current archive"). Every response claim of the paper is measured in V1, V2 and V3.

| Claim | Tested | Outcome |
|---|---|---|
| localiser-masks-ventral-dorsal | no | blocked, data missing (V1 to V3 and surfaces) |
| presentation-source-interaction | no | blocked, data missing (V1 to V3) |
| interaction-not-area-dependent | no | blocked, data missing (V1 to V3) |
| lower-field-below-greater | no | blocked, data missing (V1 to V3) |
| upper-field-no-source-difference | no | blocked, data missing (V1 to V3) |
| fig4-top-five-matching-source | no | blocked, data missing (V1 to V3) |
| fig4-qualitative-content | no | blocked, data missing (depends on fig4-top-five-matching-source) |
| fig5a-mean-luminance | no | blocked, data missing (responses; monitor calibration) |
| fig5-no-other-pixel-relations | no | blocked, data missing (as fig5a) |
| fig5bc-spatial-frequency | no | blocked, data missing (as fig5a) |
| fig5def-orientation | no | blocked, data missing (as fig5a) |
| fig5-no-other-filter-relations | no | blocked, data missing (as fig5a) |
| no-radial-bias-in-patches | no | blocked, data missing (monitor calibration; depends on fig5bc) |
| figs2-task-no-condition-difference | yes | see its section |
| figs3-residual-no-condition-difference | no | blocked, data missing (V1 to V3) |
| figs3-residual-upper-larger | no | blocked, data missing (V1 to V3) |

The blocked response claims are not tested. To describe the data set, the data check fits the paper's
GLM in the volume, and a last section repeats the response analysis on a volume localiser region that
is not V1 to V3. No outcome rests on that section.

In [ ]:
import time
import resource
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats

import vim3

pd.set_option("display.width", 200)
started = time.time()
participants, stim = [], None
for p in vim3.stream_participants():
    if p["id"] == "stim":
        stim = p
        continue
    r = vim3.process_participant(p)
    for reading in ("a", "b", "c"):
        r["perf_" + reading] = vim3.task_performance(p, reading)
    for k in ("md5", "series", "seq", "lut", "task", "affine", "scale"):
        r[k] = p[k]
    participants.append(r)
    del p
print(f"data set {vim3.DATASET}, version {vim3.VERSION}, file md5 expected {vim3.TAR_MD5}")
print(f"streamed and processed {len(participants)} participants in {time.time() - started:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB; "
      f"{stim['reconnects']} reconnections")

## Data check

In [ ]:
listed = dict(line.split()[::-1] for line in urllib.request.urlopen(
    "https://crcnsarchive.s3-us-west-2.amazonaws.com/vim-3/checksums.md5").read().decode().splitlines() if line.strip())
print("crcns_vim-3.tar.gz: md5 as streamed", stim["tar_md5"], "| in checksums.md5", listed["crcns_vim-3.tar.gz"],
      "| match:", stim["tar_md5"] == listed["crcns_vim-3.tar.gz"], "| bytes read", stim["tar_bytes"])

rows = []
for r in participants:
    kinds = Counter("nii" if n.endswith(".nii") else n.split("_")[-1].replace(".npy", "").replace("lut", "task_lut")
                    if not n.endswith("task_lut.npy") else "task_lut" for n in r["md5"])
    rows.append({"participant": r["id"], "nifti": kinds["nii"], "seq": kinds["seq"], "task": kinds["task"],
                 "task_lut": kinds["task_lut"], "series": " ".join(map(str, r["series"])),
                 "shape": "x".join(map(str, r["shape"])), "dtype": r["dtype"],
                 "raw range": f"{r['raw_range'][0]} to {r['raw_range'][1]}", "slope": round(r["scale"][0], 3),
                 "voxel mm": tuple(np.sqrt((r["affine"][:3, :3] ** 2).sum(0)).round(2)),
                 "presses": sum(len(v) for v in r["task"].values())})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
md5s = Counter(h for r in participants for h in r["md5"].values())
print("\nfiles", sum(len(r["md5"]) for r in participants), "| distinct md5", len(md5s),
      "| files with the same content as another:", sum(c for c in md5s.values() if c > 1))
seq_keys = Counter(r["seq"][k].tobytes() for r in participants for k in r["seq"])
print("run sequences", sum(seq_keys.values()), "| distinct", len(seq_keys))

The descriptor says each participant directory holds the 10 fMRI recordings with the sequence and task
files; every directory holds 10 NIfTI files, 10 sequence, 10 task and 10 task lookup files. Each NIfTI is
96 x 96 x 35 voxels by 166 volumes, 2 mm voxels, int16 with a scale factor, which matches the Methods
(matrix 96 x 96, 35 slices, 2 mm isotropic, 332 s at TR 2 s). No two files share their content, and no
run sequence is repeated, so no recording was deposited twice. The NIfTI files are named by scanner
series number, not run number; for p1003 and p1010 the series run 3, 4, 6 to 13 and 4 to 13. Run n is
taken to be the n-th series in order, which is what the participant configurations of the paper's code
list (for example p1003 `[3, 4] + range(6, 14)`); the check of the run order below tests this against
the data.

In [ ]:
r0 = participants[0]
seq = r0["seq"][1]
print("run sequence array", seq.shape, "(descriptor: 2 x 83 x 5)")
rows = []
for r in participants:
    for run in range(1, 11):
        s = r["seq"][run]
        for f, name in ((0, "upper"), (1, "lower")):
            e = s[f]
            img = e[e[:, 1] > 0]
            main = e[e[:, 3] == 0]
            rows.append({"participant": r["id"], "run": run, "field": name, "events": len(e),
                         "main events": len(main), "main null": int((main[:, 1] == 0).sum()),
                         "main above": int((main[:, 1] == 1).sum()), "main below": int((main[:, 1] == 2).sum()),
                         "main pairs": len({(int(a), int(b)) for a, b in main[main[:, 1] > 0][:, 1:3]}),
                         "null": int((e[:, 1] == 0).sum()), "above": int((e[:, 1] == 1).sum()),
                         "below": int((e[:, 1] == 2).sum()), "pre": int(e[:, 3].sum()),
                         "images": len(set(img[:, 2].astype(int))),
                         "first onset": e[0, 0], "last onset": e[-1, 0],
                         "step": float(np.unique(np.diff(e[:, 0]))[0]) if len(np.unique(np.diff(e[:, 0]))) == 1 else np.nan})
ev = pd.DataFrame(rows)
print(ev.groupby("field")[["events", "main events", "main null", "main above", "main below", "main pairs", "pre",
                          "images", "first onset", "last onset", "step"]]
      .agg(["min", "max"]).to_string())
adv = ev.pivot_table(index=["participant", "run"], columns="field", values="first onset")
print("\nruns where the lower field starts 2 s after the upper:", int((adv.lower - adv.upper == 2).sum()),
      "| upper 2 s after lower:", int((adv.upper - adv.lower == 2).sum()), "of", len(adv))
ids_seq = sorted({int(i) for r in participants for s in r["seq"].values() for i in s[..., 2].ravel() if i > 0})
ids_stim = sorted({int(n.split("_img_id_")[1][:3]) for n in stim["names"]})
print("image ids in sequences", len(ids_seq), "| in stimulus screenshots", len(ids_stim), "| same:", ids_seq == ids_stim,
      "| screenshots", len(stim["names"]))
col = pd.DataFrame(stim["colour"]).T
print("screenshot sizes", col.groupby(["width", "height"]).size().to_dict(), "| largest R-G difference", col.max_rg.max(),
      "| largest R-B difference", col.max_rb.max(), "| screenshots with R, G and B not all equal", int(((col.max_rg > 0) | (col.max_rb > 0)).sum()))
lut = pd.DataFrame([{"participant": r["id"], "run": k, "rows": len(v), "t0": v[0, 0], "t_end": v[-1, 0],
                     "rate_hz": 1 / np.median(np.diff(v[:, 0])), "digits": f"{int(v[:, 1].min())}-{int(v[:, 1].max())}",
                     "polarities": " ".join(map(str, sorted(set(v[:, 2].astype(int))))), "targets": int(v[:, 3].sum())}
                    for r in participants for k, v in r["lut"].items()])
print("\ntask lookup:", lut[["rows", "t0", "t_end", "rate_hz", "targets"]].agg(["min", "max"]).round(3).to_string())
print("digits", lut.digits.unique(), "polarities", lut.polarities.unique())
press = pd.DataFrame([{"participant": r["id"], "key": k.decode(), "time": t} for r in participants
                      for v in r["task"].values() for (k, t) in v])
print("\nbutton presses: keys", press.key.value_counts().to_dict(), "| time range", press.time.min().round(2),
      "to", press.time.max().round(2), "s")

Each field's sequence has 83 events at 4 s steps, of which 8 are marked 'pre' (the Design: 83 events,
4 s inter-event interval, the final 8 events replicated and prepended). In every run and field the
other 75 hold 15 null events and 60 image events, 30 above and 30 below, each of the 60 image and
source pairs once, as the Design says; the 30 image ids are those of the 120
screenshots (30 images by 2 sources by 2 presentation locations). The screenshots are 1920 by 1200
pixels, the display resolution of the Methods. Every screenshot holds pixels whose R, G and B values
differ (by up to 192 levels); this count covers the whole screen, overlay lines included, and the
paper's analyses of the chromatic channels of the patches show that the patches are in colour too. The
image statistics in DKL space therefore cannot be computed without the monitor calibration. In every run one field's sequence
starts 2 s after the other's, as the Design says, and which one varies by run. The task lookup has 996
digits at 3 Hz over 332 s, as the descriptor says and the Methods' 3 Hz stream. Button times lie within
the run; four different keys were used, and task.py counts any key as a response, as is done here.

The next checks fit the paper's GLM in the volume, without the paper's preprocessing (no slice timing
or motion correction, no projection to the surface) and with ordinary least squares in place of
3dREMLfit. They test that the image sequences share the clock of the fMRI volumes and that the NIfTI
order is the run order. For each run, a localiser mask (voxels with t > 2.58 for one regressor for all
image events of a field, as in the Methods, Localiser) is made from the other nine runs, and the mean
over that mask of the held-out run is fitted with: each run's sequence (run order); its own sequence
shifted by -12 to +12 s (clock); and the other field's sequence (contrast between conditions: the two
fields' events alternate every 2 s, so a voxel set chosen for one field should follow that field's
timing better than the other's).

In [ ]:
rows = []
for r in participants:
    for f, name in ((0, "upper"), (1, "lower")):
        q = r["fields"][f]
        prof = q["shift_t"].mean(0)
        rows.append({"participant": r["id"], "field": name, "mask voxels (all runs)": q["n_mask"],
                     "held-out mask voxels (min-max)": f"{q['n_mask_cv'].min()}-{q['n_mask_cv'].max()}",
                     "runs whose best sequence is their own": int((q["order_t"].argmax(1) == np.arange(10)).sum()),
                     "t own seq (mean)": q["order_t"].diagonal().mean().round(2),
                     "t other runs' seq (mean)": q["order_t"][~np.eye(10, dtype=bool)].mean().round(2),
                     "t other field (mean)": q["other_field_t"].mean().round(2),
                     "best shift s": int(q["shifts"][prof.argmax()]),
                     "t at shift 0": prof[list(q["shifts"]).index(0)].round(2),
                     "t at +-12 s": f"{prof[0]:.2f} / {prof[-1]:.2f}"})
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
print("\nmean t over participants by shift (s):")
shifts = participants[0]["fields"][0]["shifts"]
prof = pd.DataFrame({f"{r['id']} {n}": r["fields"][f]["shift_t"].mean(0) for r in participants for f, n in ((0, "U"), (1, "L"))},
                    index=shifts).T
print(prof.round(1).to_string())
print("\nhead motion index, largest centre-of-mass displacement from the first volume of run 1, mm, per run:")
print(pd.DataFrame({r["id"]: r["motion_mm"] for r in participants}, index=range(1, 11)).T.round(1).to_string())

For six of the seven participants the held-out runs follow their own sequence: the mean t for the own
sequence (2.6 to 4.9) is above the mean for the other runs' sequences, most runs fit their own sequence
best (5 to 10 of 10 per field), the fit is best at a shift of -2 or 0 s and falls to near zero or below
at 8 to 12 s, and the mask of each field follows its own field's timing better than the other field's.
This is consistent with the sequences sharing the clock of the volumes, with the first volume at time 0
of the sequence, and with the n-th series being run n. A best shift of -2 s, half the 4 s event
spacing, is within what the HRF model and the unshifted slice times allow; it is not a measured offset.
For p1011 none of this could be established: the own-sequence t is near 1, its runs do not pick their
own sequence (1 and 2 of 10), and the motion index grows to 3.6 mm by run 10, against at most 1.8 mm
in the others. Without motion correction the volume analysis cannot separate a timing or ordering
problem in p1011 from motion; the paper motion-corrected the data. The p1011 clock and run order are
therefore not shown, and this bears only on the volume variant at the end, not on the tested claim,
which uses the sequence and task files alone.

In [ ]:
rows = []
for r in participants:
    ju, jl = r["fields"][0]["mask_j"], r["fields"][1]["mask_j"]
    ku, kl = r["fields"][0]["mask_k"], r["fields"][1]["mask_k"]
    rows.append({"participant": r["id"], "upper mask voxels": len(ju), "lower mask voxels": len(jl),
                 "upper mean S-I index": ju.mean().round(1), "lower mean S-I index": jl.mean().round(1),
                 "upper below lower (mm)": round(2 * (ju.mean() - jl.mean()), 1),
                 "Mann-Whitney p": stats.mannwhitneyu(ju, jl).pvalue})
loc = pd.DataFrame(rows)
print("Axis codes of the image axes:", "".join(__import__("nibabel").aff2axcodes(participants[0]["affine"])),
      "(second axis increases toward inferior)")
print(loc.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
print("participants whose upper-field mask lies below the lower-field mask:", int((loc["upper below lower (mm)"] > 0).sum()), "of 7")

The affine puts the second image axis from superior to inferior. In six of seven participants the voxels
that respond to upper-field presentation lie on average below (inferior to) those that respond to
lower-field presentation, by 1.6 to 13.7 mm, which is the retinotopic expectation (upper field on the
ventral bank) and a second contrast between conditions; p1011 is again the exception. This is a volume
check over the whole slab, not the paper's test of the localiser claim, which needs V1 to V3 on the
cortical surface.

## figs2-task-no-condition-difference

> Performance on the central fixation task does not differ across the presentation and source location
> conditions. (Results and Discussion, paragraph 7; Figure S2 legend)

Criterion (claims file): performance as in the Figure S2 legend and task.py, the correlation of target
presence assigned to a condition with button responses at lags 0 to 1.9 s in 0.1 s bins, averaged over
lags 0.4 to 0.9 s (bins 4 to 9), and a two-way repeated measures ANOVA (presentation by source), as
group.py tasks(). Reproduced when no effect has p < 0.05. Reading (a) is task.py as written, reading (b)
assigns targets by absolute run time; both keep the p1004 rule. Variant (c), added here for information,
bins the whole of 32 to 332 s (task.py puts every press and target after 300 s into its last bin). With
two levels per factor each ANOVA effect is a paired contrast over the seven participants, F(1, 6) = t².

In [ ]:
names = {(0, 0): "upper/above", (0, 1): "upper/below", (1, 0): "lower/above", (1, 1): "lower/below"}
results = {}
for reading in ("a", "b", "c"):
    perf = np.array([r["perf_" + reading][0] for r in participants])  # (7, pres, src, lag)
    win = perf[..., 4:10].mean(-1)
    results[reading] = (perf, win, vim3.rm_anova_2x2(win))
    print(f"Reading ({reading}): window 0.4-0.9 s, correlation per participant and condition")
    print(pd.DataFrame({names[k]: win[:, k[0], k[1]] for k in names}, index=[r["id"] for r in participants]).round(4).to_string())
    print("  mean", {names[k]: round(float(win[:, k[0], k[1]].mean()), 4) for k in names})
    for eff, (F, d1, d2, p) in results[reading][2].items():
        print(f"  {eff:9s} F({d1},{d2}) = {F:.3f}, p = {p:.3f}")
    counts = pd.DataFrame([r["perf_" + reading][1] for r in participants], index=[r["id"] for r in participants]).fillna(0).astype(int)
    print("  targets after 32 s, and how they were assigned (totals):", counts.sum().to_dict())
    print()

In [ ]:
perf = results["a"][0]
lag_mean = perf.mean(0)  # (pres, src, lag)
print("Reading (a): mean correlation over participants by lag (s); rows are conditions")
print(pd.DataFrame({names[k]: lag_mean[k[0], k[1]] for k in names}, index=np.arange(20) / 10).T.round(3).to_string())
print("peak lag per condition (s):", {names[k]: float(lag_mean[k[0], k[1]].argmax() / 10) for k in names})
print("range over all participants, conditions and lags:", perf.min().round(3), "to", perf.max().round(3))
# the window is a choice of group.py: show other windows
for lo, hi in ((3, 8), (4, 10), (5, 11), (6, 10)):
    for reading in ("a", "b"):
        w = results[reading][0][..., lo:hi].mean(-1)
        st = vim3.rm_anova_2x2(w)
        print(f"window {lo/10:.1f}-{(hi-1)/10:.1f} s, reading ({reading}):",
              ", ".join(f"{e} F = {v[0]:.2f}, p = {v[3]:.3f}" for e, v in st.items()))
# the effect sizes: largest condition difference in the window, as a fraction of the mean
for reading in ("a", "b", "c"):
    w = results[reading][1].mean(0)
    print(f"reading ({reading}): condition means {w.ravel().round(4)}, range / mean = {(w.max() - w.min()) / w.mean():.3f}")

**Result: reproduced.** No effect of presentation, source or their interaction reaches p < 0.05 under
either reading the criterion names, or under variant (c), and the response peaks at the same lag for
the four conditions (0.6 s). The condition means in the window differ by less than a tenth of their mean
(range over mean 0.046 to 0.060). Of the other windows shown, one gives a presentation effect under
reading (a) (0.6 to 0.9 s, F = 7.51, p = 0.034); none does under reading (b), and the claim's window is
that of group.py. Under reading (a) the assignment compares onsets 32 s
away from the target, so its condition labels are close to random; reading (b) is the one the legend
describes, and it gives the same outcome.

## Volume variant of the response claims (no outcome rests on it)

The response claims need V1, V2 and V3, which are not in the data set, and they are blocked. To show
what the data set can support, the paper's response analysis (Methods, Response amplitude) is repeated
here on the volume localiser masks of the data check: for each participant and field, the mean time
course over all voxels with t > 2.58, a GLM with one regressor per image and source (60 per field), and
psc from the polynomial baseline. These masks are not V1 to V3: they include any responsive voxel in the
35-slice occipital slab, the data are not motion corrected and OLS replaces 3dREMLfit, and there is no
area factor. The numbers below describe the data set and are not compared with the claims' tolerances.

In [ ]:
labels = participants[0]["fields"][0]["labels"]
assert all(r["fields"][f]["labels"] == labels for r in participants for f in (0, 1))
A = np.array([[r["fields"][f]["psc_image"] for f in (0, 1)] for r in participants]).reshape(7, 2, 30, 2)
m = A.mean(2)  # participant, pres (upper, lower), src (above, below)
print("psc per participant (mean over 30 images):")
print(pd.DataFrame({names[k]: m[:, k[0], k[1]] for k in names}, index=[r["id"] for r in participants]).round(3).to_string())
print("mean over participants:", {names[k]: round(float(m[:, k[0], k[1]].mean()), 3) for k in names})
print("range of all per-image psc values:", A.min().round(2), "to", A.max().round(2))
for label, sub in (("all 7", slice(None)), ("without p1011", [i for i, r in enumerate(participants) if r["id"] != "p1011"])):
    mm = m[sub]
    an = vim3.rm_anova_2x2(mm)
    tl = stats.ttest_rel(mm[:, 1, 1], mm[:, 1, 0])
    tu = stats.ttest_rel(mm[:, 0, 1], mm[:, 0, 0])
    print(f"\n{label}: presentation x source F(1,{an['pres:src'][2]}) = {an['pres:src'][0]:.2f}, p = {an['pres:src'][3]:.3f}"
          f" | source F = {an['src'][0]:.2f}, p = {an['src'][3]:.3f} | presentation F = {an['pres'][0]:.2f}, p = {an['pres'][3]:.3f}")
    print(f"  lower field, below vs above: {mm[:, 1, 1].mean():.3f} vs {mm[:, 1, 0].mean():.3f} psc, t = {tl.statistic:.2f}, p = {tl.pvalue:.3f}")
    print(f"  upper field, below vs above: {mm[:, 0, 1].mean():.3f} vs {mm[:, 0, 0].mean():.3f} psc, t = {tu.statistic:.2f}, p = {tu.pvalue:.3f}")
d = (A[:, 0] - A[:, 1]).mean(0).reshape(60)  # upper minus lower, mean over participants
order = np.argsort(d)
lab = np.array([f"{i:03d} {s}" for i, s in labels])
print("\nfive most upper-preferred:", list(zip(lab[order[::-1][:5]], d[order[::-1][:5]].round(3))))
print("five most lower-preferred:", list(zip(lab[order[:5]], d[order[:5]].round(3))))
src = np.array([s for _, s in labels])
print("above among top five upper-preferred:", int((src[order[::-1][:5]] == "above").sum()),
      "| below among top five lower-preferred:", int((src[order[:5]] == "below").sum()))
rm = np.array([[r["fields"][f]["resid_ms"] for f in (0, 1)] for r in participants])
print("\nmean squared residual of the ROI GLM, psc^2, upper and lower:", rm.round(3).tolist())
print("upper larger in", int((rm[:, 0] > rm[:, 1]).sum()), "of 7; paired t =", round(stats.ttest_rel(rm[:, 0], rm[:, 1]).statistic, 2),
      "p =", round(stats.ttest_rel(rm[:, 0], rm[:, 1]).pvalue, 3))

In this volume substitute, lower-field responses are larger for patches from below fixation than from
above, the upper-field responses do not differ by source, and the most lower-preferred patch pairs are
mostly from below fixation (5 of 5, and 4 of 5 upper-preferred from above), the pattern the paper
reports; the presentation by source interaction is weaker than the paper's (p = 0.137). The residual of
the region's GLM is larger for upper presentation in 5 of 7 participants, but not on average, because of
p1011, whose region is large and whose residuals are an order of magnitude larger than the others'. The printed numbers are the record of this; none of them is an outcome, since
the region is not the paper's.

## Summary

| Claim | Data | Outcome |
|---|---|---|
| localiser-masks-ventral-dorsal | dataset | blocked: data missing |
| presentation-source-interaction | dataset | blocked: data missing |
| interaction-not-area-dependent | dataset | blocked: data missing |
| lower-field-below-greater | dataset | blocked: data missing |
| upper-field-no-source-difference | dataset | blocked: data missing |
| fig4-top-five-matching-source | dataset | blocked: data missing |
| fig4-qualitative-content | with inputs | blocked: data missing |
| fig5a-mean-luminance | with inputs | blocked: data missing |
| fig5-no-other-pixel-relations | with inputs | blocked: data missing |
| fig5bc-spatial-frequency | with inputs | blocked: data missing |
| fig5def-orientation | with inputs | blocked: data missing |
| fig5-no-other-filter-relations | with inputs | blocked: data missing |
| no-radial-bias-in-patches | with inputs | blocked: data missing |
| figs2-task-no-condition-difference | dataset | reproduced |
| figs3-residual-no-condition-difference | dataset | blocked: data missing |
| figs3-residual-upper-larger | dataset | blocked: data missing |

How the analysis differs from the paper's. The tested claim follows task.py and group.py, with the
ANOVA computed as paired contrasts (exact for a 2 x 2 design) in place of the code's `fmri_tools`
routine. The volume checks and variant use OLS without slice timing or motion correction, in the
volume, without the V1 to V3 restriction.

What was found about the data set. The archive holds the functional runs, event sequences and task
files of all seven participants, with no duplicated recording, and its file matches its listed md5.
It lacks the V1, V2 and V3 definitions, the cortical surfaces and anatomical images, which 14 of the 16
claims need directly or through the claims they depend on, and the monitor calibration needed for the
image statistics is not public, so 15 claims are blocked. The NIfTI files are named by scanner series, not run, and the descriptor does
not give the mapping; the paper's code does, and the data support it for six participants. For p1011
the clock and run order could not be confirmed in uncorrected volume data, and that participant moves
more than the others.